# 143 -- Context Compression

**What you'll learn:**
- Why context compression matters: cost, latency, and attention dilution
- Extractive vs abstractive compression -- how they differ and when to choose each
- Relevance scoring approaches: TF-IDF, embedding similarity, and LLM judge
- The compress->answer pipeline as an ASCII flow diagram
- Token counting: why `len(text) // 4` is a rough estimate and when `tiktoken` helps
- Part A (no API key needed) vs Part B (`OPENAI_API_KEY` required)

```
Source: examples/143-context-compression/  |  Model: gpt-4o-mini  |  Key: OPENAI_API_KEY
```

In [ ]:
%pip install -q openai python-dotenv langgraph

---
## Part A -- CPU-Safe Concept Demonstrations

All cells in Part A run without any API key. They use the same helper functions as `src/tools.py` but with hardcoded mock scores so you can explore the pipeline logic before making live calls.

### Why Context Compression Matters

LLMs charge per token and generate slower as context grows. Here is what sending a document at full size costs:

| Document size | Tokens sent (full) | Cost at \$0.15/MTok | Latency estimate |
|---|---|---|---|
| 500 words | ~625 | \$0.00009 | ~500 ms |
| 2,000 words | ~2,500 | \$0.000375 | ~900 ms |
| 10,000 words | ~12,500 | \$0.001875 | ~2,500 ms |
| 100,000 words | ~125,000 | \$0.01875 | ~15,000 ms |

At scale (millions of requests), those fractions of a cent add up fast.

**Attention dilution** is a subtler problem: transformer attention is finite. When the context contains thousands of irrelevant tokens, the model must compete between relevant evidence and noise. Studies show answer quality degrades measurably past ~8k tokens of unrelated content, even when the relevant passage is present. Compression removes the noise *before* it reaches the model.

### Extractive vs Abstractive Compression

There are two philosophies for making a document shorter:

| Type | How | Example output | Pros | Cons |
|---|---|---|---|---|
| Extractive | Select sentences verbatim | "CPython is an interpreter. The GIL prevents parallel execution." | Faithful, no hallucination | Choppy, loses flow |
| Abstractive | Summarize / paraphrase | "CPython interprets code via bytecode; the GIL limits thread parallelism." | Fluent, compact | Can hallucinate |
| Hybrid | Extract + rewrite key passages | Best of both | Complex pipeline | |

**This example uses extractive compression** -- sentences are selected verbatim based on a relevance score, then joined. No rewriting means no hallucination risk.

### Relevance Scoring Approaches

The heart of any compression pipeline is the scorer -- the function that assigns a 0-1 relevance score to each sentence.

| Method | How | Cost | Quality | Use when |
|---|---|---|---|---|
| TF-IDF | Keyword overlap between sentence and query | Free | Keyword-sensitive only | Simple search |
| Embedding similarity | Cosine of sentence embedding vs query embedding | Low (embed once) | Semantic -- catches synonyms | Production |
| LLM judge | Ask LLM to rate each sentence 0-1 | Medium (extra LLM call) | Best quality | When accuracy matters |
| BM25 | Improved TF-IDF with length normalization | Free | Medium | Fast retrieval |

**This example uses an LLM judge** (via `score_sentences()` in `src/tools.py`) -- one prompt asks the model to rate all sentences at once, which is more accurate than keyword overlap and only costs one extra API call.

### The Compress->Answer Pipeline

The full pipeline from raw document to final answer:

```
Document
   |
   v
split_sentences() -> [s1, s2, s3, ... sN]
   |
   v
score_sentences(query) -> [(0.9, s4), (0.7, s2), (0.3, s1), ...]
   |
   v
keep top ratio% -> compressed_context
   |
   v
answer_llm(compressed_context, query) -> answer
```

**The scoring tradeoff:** The scoring step itself uses LLM tokens (or embedding calls). The pipeline breaks even when the document is long enough that savings on the answer call exceed the scoring cost. For a 500-sentence document with a 0.4 ratio:
- Tokens saved on answer: ~60% of document tokens
- Tokens spent on scoring: ~1x document tokens (one scoring call)
- Net positive from the **second query onward** (first query pays for scoring)

### Token Counting: `chars / 4` vs `tiktoken`

The estimate `len(text) // 4` works because GPT tokenizers average ~4 characters per token for typical English prose.

**When it fails:**
- **CJK text** (Chinese, Japanese, Korean) -- each character is typically 1 token but only 1 character, so `chars/4` underestimates by ~4x
- **Source code** -- identifiers and keywords tokenize efficiently; code often has ~2-3 chars/token rather than 4
- **Very short strings** -- rounding errors dominate at <20 chars
- **Emoji** -- a single emoji can be 1-3 tokens despite being 1-4 chars

**`tiktoken`** gives the exact token count used by OpenAI models and is the right tool when:
- You are enforcing a hard context window limit
- You are billing users per token
- The text might contain CJK, emoji, or unusual characters

For this workbook, `count_tokens()` = `len(text) // 4` -- sufficient for benchmarking ratios and comparing before/after.

In [ ]:
# Part A -- split_sentences() demo
# Exact function from src/tools.py -- no LLM needed.

import re

def split_sentences(text: str) -> list[str]:
    """Split text into sentences on sentence-ending punctuation."""
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def count_tokens(text: str) -> int:
    """Rough token estimate: ~4 chars per token."""
    return len(text) // 4


SAMPLE_PARAGRAPH = (
    "The Python programming language was created by Guido van Rossum and first released in 1991. "
    "Python's design philosophy emphasizes code readability with the use of significant indentation. "
    "It supports multiple programming paradigms, including structured, object-oriented, and functional programming. "
    "CPython is the reference implementation of Python. "
    "The GIL in CPython prevents true parallel execution of Python threads. "
    "Python is dynamically typed and garbage-collected. "
    "It features a comprehensive standard library often described as batteries included."
)

sentences = split_sentences(SAMPLE_PARAGRAPH)

print(f"Input: {len(SAMPLE_PARAGRAPH)} chars, ~{count_tokens(SAMPLE_PARAGRAPH)} tokens")
print(f"Sentences found: {len(sentences)}")
print()
for i, s in enumerate(sentences):
    print(f"  [{i+1}] ({count_tokens(s):>3} tok) {s}")

In [ ]:
# Part A -- count_tokens() demo
# Shows where chars/4 holds and where it breaks down.

examples = [
    ("English prose",   "The quick brown fox jumps over the lazy dog."),
    ("Python code",     "def fib(n): return n if n<=1 else fib(n-1)+fib(n-2)"),
    ("JSON data",       '{"key": "value", "nested": {"a": 1, "b": true}}'),
    ("URL string",      "https://api.example.com/v1/endpoint?param=value&other=123"),
    ("Short text",      "Yes."),
]

print(f"{'Text type':<16} | {'Chars':>6} | {'Chars/4 est':>11} | Notes")
print("-" * 68)
for name, text in examples:
    chars = len(text)
    est = chars // 4
    note = ""
    if "code" in name.lower():
        note = "<-- may underestimate (code tokenizes densely)"
    elif name == "Short text":
        note = "<-- rounding error dominates at <20 chars"
    elif "URL" in name:
        note = "<-- slashes/dots each often 1 token"
    print(f"{name:<16} | {chars:>6} | {est:>11} | {note}")

print()
print("Note: CJK text (Chinese/Japanese/Korean) gives ~1 char per token,")
print("so chars/4 underestimates by ~4x. Use tiktoken for non-ASCII content.")
print()
print("Takeaway: chars/4 is +-15% for English prose -- good enough for ratio benchmarking.")

In [ ]:
# Part A -- tiktoken vs chars/4 comparison (optional: runs only if tiktoken is installed)
# Install with: pip install tiktoken

try:
    import tiktoken
    enc = tiktoken.encoding_for_model("gpt-4o")
    HAS_TIKTOKEN = True
except ImportError:
    HAS_TIKTOKEN = False

test_strings = [
    ("English prose",  "The Python programming language emphasizes readability above all else."),
    ("Python code",    "def fib(n): return n if n<=1 else fib(n-1)+fib(n-2)"),
    ("Dense JSON",     '{"key":"val","n":1,"b":true,"arr":[1,2,3]}'),
    ("Long URL",       "https://api.openai.com/v1/chat/completions?model=gpt-4o-mini"),
    ("Short text",     "Yes."),
]

if HAS_TIKTOKEN:
    print(f"{'Text type':<16} | {'Chars/4':>7} | {'Tiktoken':>8} | {'Error %':>8}")
    print("-" * 50)
    for name, text in test_strings:
        est   = len(text) // 4
        exact = len(enc.encode(text))
        err   = (est - exact) / exact * 100
        print(f"{name:<16} | {est:>7} | {exact:>8} | {err:>+7.1f}%")
    print()
    print("chars/4 is most accurate for English prose (~0-15% error).")
    print("Code and JSON can differ by 20-40% because tokens are denser.")
else:
    print("tiktoken not installed -- showing chars/4 estimates only")
    print("Install with: pip install tiktoken")
    for name, text in test_strings:
        print(f"  {name:<16}: ~{len(text) // 4} tokens (chars/4 estimate)")

In [ ]:
# Part A -- score sentences with MOCK scores (no LLM)
# Hardcoded relevance scores for query: "How does CPython execute Python code?"

QUERY = "How does CPython execute Python code?"

# Pretend an LLM rated each sentence 0.0 (irrelevant) to 1.0 (highly relevant)
MOCK_SCORES = [
    0.10,  # sentence 1 -- creation by Guido -- not about execution
    0.05,  # sentence 2 -- design philosophy -- not about execution
    0.15,  # sentence 3 -- programming paradigms -- tangentially relevant
    0.90,  # sentence 4 -- CPython is reference implementation -- very relevant
    0.85,  # sentence 5 -- GIL prevents parallel threads -- relevant to execution
    0.20,  # sentence 6 -- dynamically typed -- somewhat relevant
    0.05,  # sentence 7 -- standard library -- not about execution
]

assert len(MOCK_SCORES) == len(sentences)

scored = list(zip(MOCK_SCORES, sentences))
scored_sorted = sorted(scored, key=lambda x: x[0], reverse=True)

ratio = 0.4
keep_n = max(1, int(len(scored_sorted) * ratio))
kept_set = {s for _, s in scored_sorted[:keep_n]}

print(f"Query: {QUERY}")
print(f"Ratio: {ratio}  ->  keep top {keep_n} of {len(scored_sorted)} sentences")
print()
print(f"{'Score':>6}  {'Status':<5}  Sentence")
print("-" * 75)
for score, sentence in scored_sorted:
    status = "KEEP" if sentence in kept_set else "DROP"
    print(f"  {score:.2f}  [{status}]  {sentence[:68]}")

In [ ]:
# Part A -- ratio sweep
# Shows how many sentences survive and the token reduction at each ratio.

original_tokens = count_tokens(SAMPLE_PARAGRAPH)

print(f"Original: {len(sentences)} sentences, ~{original_tokens} tokens")
print()
print(f"{'Ratio':>7} | {'Sentences kept':>14} | {'Approx tokens':>14} | {'Reduction':>10}")
print("-" * 55)

for ratio in [1.0, 0.6, 0.4, 0.2]:
    keep_n = max(1, int(len(scored_sorted) * ratio))
    kept_sentences = [s for _, s in scored_sorted[:keep_n]]
    compressed = " ".join(kept_sentences)
    tok = count_tokens(compressed)
    pct = round((1 - tok / original_tokens) * 100) if original_tokens else 0
    print(f"{ratio:>7.1f} | {keep_n:>6}/{len(sentences):<7} | {tok:>14} | {pct:>9}%")

In [ ]:
# Part A -- ratio sweep on a longer 10-sentence document
# Verifies the quality-vs-ratio table above with real token numbers.

import re

LONGER_DOC = (
    "Python is a high-level, general-purpose programming language. "
    "Its design philosophy emphasizes code readability and simplicity. "
    "Python is dynamically typed and garbage-collected. "
    "It supports multiple programming paradigms including structured, OOP, and functional. "
    "Guido van Rossum created Python and first released it in 1991. "
    "Python consistently ranks as one of the most popular programming languages. "
    "The language makes it popular for rapid application development. "
    "Python has a comprehensive standard library often described as batteries-included. "
    "It is commonly used in web development, scientific computing, AI, and scripting. "
    "Python 3.0, released in 2008, was a major revision not fully backward-compatible with Python 2."
)

long_sentences = split_sentences(LONGER_DOC)

# Mock scores for query: "What makes Python popular for programming?"
long_scores = [0.9, 0.8, 0.5, 0.6, 0.4, 0.95, 0.85, 0.7, 0.88, 0.3]

long_scored = sorted(zip(long_scores, long_sentences), reverse=True)
orig_tok = count_tokens(LONGER_DOC)

print(f"Original: {len(long_sentences)} sentences, ~{orig_tok} tokens")
print(f"Query: 'What makes Python popular for programming?'")
print()
print(f"{'Ratio':>7} | {'Sentences':>9} | {'Est. tokens':>12} | {'Token savings':>14}")
print("-" * 50)

for ratio in [1.0, 0.8, 0.6, 0.4, 0.2]:
    keep_n = max(1, int(len(long_scored) * ratio))
    kept   = [s for _, s in long_scored[:keep_n]]
    tok    = count_tokens(" ".join(kept))
    savings = (orig_tok - tok) / orig_tok * 100
    print(f"{ratio:>7.1f} | {len(kept):>9} | {tok:>12} | {savings:>13.0f}%")

### Quality vs Compression Ratio

There is no free lunch -- aggressive compression always risks removing relevant information.

| Ratio | Tokens kept | Quality impact | When to use |
|---|---|---|---|
| 1.0 (no compression) | 100% | None | Debugging, high-stakes queries |
| 0.8 | 80% | Negligible | Remove clearly irrelevant boilerplate |
| 0.5 | 50% | Moderate | Good default for multi-doc RAG |
| 0.3 | 30% | Noticeable | Useful when context window is the constraint |
| 0.1 | 10% | High | Extreme budget mode -- accept quality drop |

> **Practical tip:** Start at 0.5 and measure answer quality against a holdout set.
> Only go lower if the context window or cost forces it.

**Common failure modes at aggressive ratios:**
- Low-threshold scores treat all sentences as equally relevant -- no compression effect
- High-threshold scores remove everything -- model answers "I don't know"
- Query too broad ("tell me about Python") -- many sentences score equally, random selection dominates

In [ ]:
# Part A -- answer quality intuition at each ratio
# Which information survives, and can we still answer the query?

QUERY = "How does CPython execute Python code?"

for ratio in [1.0, 0.6, 0.4, 0.2]:
    keep_n = max(1, int(len(scored_sorted) * ratio))
    kept_sentences = [s for _, s in scored_sorted[:keep_n]]
    dropped_sentences = [s for _, s in scored_sorted[keep_n:]]

    cpython_mentioned = any("CPython" in s for s in kept_sentences)
    can_answer = cpython_mentioned

    def short(s: str, n: int = 50) -> str:
        return s[:n] + "..." if len(s) > n else s

    print(f"Ratio {ratio:.1f} -- kept {keep_n}/{len(scored_sorted)} sentences")
    print(f"  Kept:    {[short(s) for s in kept_sentences]}")
    print(f"  Dropped: {[short(s, 40) for s in dropped_sentences]}")
    verdict = "YES" if can_answer else "NO -- CPython not in context"
    print(f"  Can answer the query? {verdict}")
    print()

### Exercise 1: Remove Short / Stopword Sentences

The current compress function keeps sentences purely by score. A sentence like `"Python is fast."` (3 words) can score highly on a broad query but contributes little information.

**Exercise:** Add a `min_words` parameter that drops sentences shorter than `N` words *before* the ratio-based selection. Short sentences get filtered out regardless of their relevance score.

```python
def compress_with_min_words(
    sentences: list[str],
    scores: list[float],
    ratio: float = 0.4,
    min_words: int = 5,
) -> list[str]:
    # TODO: filter sentences with fewer than min_words words
    # then sort by score and keep top ratio%
    pass
```

In [ ]:
# Part A -- multi-document compression demo (no LLM)
# Shows how to compress and merge multiple documents for a single query.
# In production, each doc is compressed independently, then re-ranked globally.

from typing import List

def mock_score(sentence: str, query: str) -> float:
    """Simple mock scorer: fraction of query words present in sentence."""
    q_words = set(query.lower().split())
    s_words = set(sentence.lower().split())
    return len(q_words & s_words) / len(q_words) if q_words else 0.5

def compress_multi_doc(
    docs: List[str],
    query: str,
    per_doc_ratio: float = 0.5,
    max_total_sentences: int = 8,
) -> str:
    """
    Compress each document independently, then merge and trim
    to max_total_sentences using global score ranking.
    """
    all_scored = []
    for doc_idx, doc in enumerate(docs):
        sents = split_sentences(doc)
        for s in sents:
            all_scored.append((mock_score(s, query), doc_idx, s))

    # Sort globally by score, keep top max_total_sentences
    all_scored.sort(reverse=True)
    top = all_scored[:max_total_sentences]
    # Restore reading order: sort by (doc_idx, original position in doc)
    top.sort(key=lambda x: (x[1], docs[x[1]].find(x[2])))
    return "\n".join(f"[doc{di}] {s}" for _, di, s in top)


DOCS = [
    "Python is a high-level language. It supports OOP and functional programming. Python is widely used in data science.",
    "Java is statically typed. It runs on the JVM. Java is used in enterprise applications and Android development.",
    "JavaScript runs in browsers. Node.js brought JavaScript to servers. It is the language of the web.",
]

MULTI_QUERY = "which language is used for data science?"
result = compress_multi_doc(DOCS, query=MULTI_QUERY, max_total_sentences=4)

print(f"Query: {MULTI_QUERY}")
print()
print("Compressed multi-doc context (top 4 sentences globally):")
print(result)

### The LangGraph Workflow: Two Nodes, One Edge

The `create_workflow()` in `src/workflow.py` is a minimal two-node LangGraph graph:

```
compress_node  -->  answer_node  -->  END
```

**`compress_node`** (calls `score_sentences` + `compress`):
- Input state: `chunks`, `query`, `ratio`
- Output: `original_tokens`, `compressed_context`, `compressed_tokens`

**`answer_node`** (calls the answering LLM):
- Input state: `compressed_context`, `query`
- Output: `answer`

Both nodes receive the OpenAI `client` and `model` via `config["configurable"]` — this keeps the workflow portable across different clients and models without changing the graph definition.

**Why LangGraph for such a simple pipeline?** For two nodes it is arguably overkill. The pattern pays off when you add branching: e.g., a node that checks whether compression was even necessary (short documents), or a retry node that falls back to the full context when the compressed answer scores too low on a confidence check.

In [ ]:
# Part A -- break-even analysis: when does compression save tokens overall?
# The scoring call costs tokens too. This cell shows at what document size
# the answer-call savings exceed the scoring cost.

# Assumptions (gpt-4o-mini pricing as of mid-2025):
# - Scoring prompt: 1x document tokens (sent once to rate sentences)
# - Answer with full context: 1x document tokens
# - Answer with compressed context: ratio * document tokens
# - If the same query is asked N times, the scoring cost is amortized over N calls.

print("Break-even analysis: does compression save tokens?")
print("(Assumes ratio=0.4, scoring costs 1x doc tokens)")
print()
print(f"{'Doc tokens':>10} | {'Score cost':>10} | {'Per-query saving':>16} | {'Queries to break even':>21}")
print("-" * 65)

ratio = 0.4
for doc_tokens in [100, 250, 500, 1000, 2500, 5000, 10000]:
    score_cost   = doc_tokens            # one scoring call = full doc
    saving_each  = doc_tokens * (1 - ratio)  # tokens saved per answer call
    break_even   = score_cost / saving_each if saving_each > 0 else float("inf")
    print(f"{doc_tokens:>10} | {score_cost:>10} | {saving_each:>16.0f} | {break_even:>21.1f}")

print()
print("For a 500-token doc: break-even at ~1.7 queries.")
print("For a 5000-token doc: break-even at ~1.7 queries (scoring cost scales with doc).")
print("Compression always pays off after ~2 queries on the same compressed context.")

In [ ]:
# Exercise 1 -- Answer Key

def compress_with_min_words(
    sentences: list[str],
    scores: list[float],
    ratio: float = 0.4,
    min_words: int = 5,
) -> list[str]:
    # Step 1: filter out sentences below the word threshold
    candidates = [
        (s, sc) for s, sc in zip(sentences, scores)
        if len(s.split()) >= min_words
    ]
    # Step 2: sort survivors by relevance score, descending
    candidates.sort(key=lambda x: x[1], reverse=True)
    # Step 3: keep the top ratio fraction of survivors
    keep_n = max(1, int(len(candidates) * ratio))
    kept_set = {s for s, _ in candidates[:keep_n]}
    # Preserve original sentence order for coherent reading
    return [s for s in sentences if s in kept_set]


# Demo: same paragraph, same mock scores, min_words=8
print("=== Without min_words filter ===")
no_filter = [
    s for _, s in sorted(zip(MOCK_SCORES, sentences), reverse=True)[:max(1, int(len(sentences)*0.4))]
]
for s in no_filter:
    print(f"  ({len(s.split())} words) {s}")

print()
print("=== With min_words=8 filter ===")
filtered = compress_with_min_words(sentences, MOCK_SCORES, ratio=0.4, min_words=8)
for s in filtered:
    print(f"  ({len(s.split())} words) {s}")

print()
print("Short sentences (< 8 words) are excluded before ratio selection.")

### Exercise 2: Add a Minimum Sentences Floor

At `ratio=0.1` on a 3-sentence document, `int(3 * 0.1) = 0` -- the compressor would keep *nothing*. The LLM would receive an empty context and answer "I don't know."

**Exercise:** Add a `min_sentences` floor so the context never drops below that count, regardless of how aggressive the ratio is.

```python
def compress_with_floor(
    sentences: list[str],
    scores: list[float],
    ratio: float = 0.4,
    min_sentences: int = 3,
) -> list[str]:
    # TODO: keep max(min_sentences, int(len * ratio)) sentences
    pass
```

In [ ]:
# Exercise 2 -- Answer Key

def compress_with_floor(
    sentences: list[str],
    scores: list[float],
    ratio: float = 0.4,
    min_sentences: int = 3,
) -> list[str]:
    scored = list(zip(scores, sentences))
    scored.sort(key=lambda x: x[0], reverse=True)
    # Apply the floor: never keep fewer than min_sentences
    keep_n = max(min_sentences, int(len(scored) * ratio))
    # But don't exceed the total number of sentences
    keep_n = min(keep_n, len(scored))
    kept_set = {s for _, s in scored[:keep_n]}
    # Return in original order
    return [s for s in sentences if s in kept_set]


# Demo: tiny 3-sentence doc, ratio=0.05 (would keep 0 without floor)
tiny_sentences = [
    "CPython compiles Python source code to bytecode.",
    "The bytecode is then interpreted by the Python virtual machine.",
    "The GIL ensures only one thread runs bytecode at a time.",
]
tiny_scores = [0.9, 0.85, 0.8]

print("=== 3-sentence doc, ratio=0.05 (without floor) ===")
raw_keep = max(1, int(len(tiny_sentences) * 0.05))
print(f"  int(3 * 0.05) = {int(3*0.05)} -> clamped to max(1, ...) = {raw_keep} sentence")

print()
print("=== With min_sentences=3 floor ===")
result = compress_with_floor(tiny_sentences, tiny_scores, ratio=0.05, min_sentences=3)
for s in result:
    print(f"  {s}")
print(f"  -> kept all {len(result)} sentences despite ratio=0.05")

---
## Part B -- Live OpenAI API Calls

**Requirements:** `OPENAI_API_KEY` in your `.env` file or shell environment.

What Part B does:
1. Compresses a ~500-word Python docs section using `src/tools.py` (`gpt-4o-mini` as the scorer)
2. Answers the same query with both full and compressed contexts
3. Runs the complete `create_workflow()` from `src/workflow.py` and prints the comparison table

**Estimated cost:** ~\$0.0005 for this demo (scoring + two answer calls).

In [ ]:
# Part B setup -- fail-fast OPENAI_API_KEY check
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY is not set.\n"
        "Add it to your .env file or run: export OPENAI_API_KEY=sk-...\n"
        "Cells in Part B will fail without it."
    )
print("OPENAI_API_KEY loaded.")

In [ ]:
# Part B -- compress a ~500-word Python docs section at ratio=0.4
# Uses compress() from src/tools.py directly.

import sys
sys.path.insert(0, ".")

from openai import OpenAI
from src.tools import compress, count_tokens

SAMPLE_CHUNKS = [
    "The Python programming language was created by Guido van Rossum and first released in 1991. "
    "Python's design philosophy emphasizes code readability with the use of significant indentation. "
    "It supports multiple programming paradigms, including structured, object-oriented, and functional programming.",

    "Python is dynamically typed and garbage-collected. "
    "It features a comprehensive standard library often described as batteries included. "
    "Python consistently ranks as one of the most popular programming languages worldwide. "
    "The language's simplicity makes it popular for rapid application development.",

    "CPython is the reference implementation of Python. "
    "When people refer to Python they often mean CPython. "
    "Other implementations include PyPy, Jython, and IronPython. "
    "CPython compiles Python source code to bytecode before executing it via the Python virtual machine. "
    "The GIL in CPython prevents true parallel execution of Python threads.",

    "Python 3 was released in 2008 and is incompatible with Python 2 in several ways. "
    "Python 2 reached end-of-life on January 1, 2020. "
    "The transition from Python 2 to Python 3 took the community over a decade to complete. "
    "Most major libraries have since dropped Python 2 support.",

    "Python's async/await syntax was introduced in Python 3.5 via PEP 492. "
    "asyncio is the standard library module for writing concurrent code with async/await. "
    "It uses an event loop to manage I/O-bound tasks without threading. "
    "This makes Python suitable for high-performance network servers and web applications.",
]

QUERY = "How does CPython execute Python code?"

client = OpenAI(api_key=OPENAI_API_KEY)

original_text = " ".join(SAMPLE_CHUNKS)
original_tokens = count_tokens(original_text)

print(f"Query: {QUERY}")
print(f"Original: {original_tokens} tokens across {len(SAMPLE_CHUNKS)} chunks")
print()
print("Scoring sentences... (one LLM call)")

compressed_text = compress(SAMPLE_CHUNKS, QUERY, client, ratio=0.4)
compressed_tokens = count_tokens(compressed_text)
reduction = round((1 - compressed_tokens / original_tokens) * 100)

print(f"Compressed: {compressed_tokens} tokens ({reduction}% reduction)")
print()
print("=== Compressed context ===")
print(compressed_text)

In [ ]:
# Part B -- ask the question with FULL context

resp_full = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": f"Context:\n{original_text}"},
        {"role": "user", "content": QUERY},
    ],
    max_tokens=256,
)

full_answer = resp_full.choices[0].message.content
full_in_tokens = resp_full.usage.prompt_tokens

print("=== Answer from FULL context ===")
print(f"Prompt tokens used: {full_in_tokens}")
print()
print(full_answer)

In [ ]:
# Part B -- ask the same question with COMPRESSED context

resp_compressed = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": f"Context:\n{compressed_text}"},
        {"role": "user", "content": QUERY},
    ],
    max_tokens=256,
)

compressed_answer = resp_compressed.choices[0].message.content
comp_in_tokens = resp_compressed.usage.prompt_tokens

print("=== Answer from COMPRESSED context ===")
print(f"Prompt tokens used: {comp_in_tokens} (was {full_in_tokens} -- saved {full_in_tokens - comp_in_tokens})")
print()
print(compressed_answer)
print()
print("--- Qualitative comparison ---")
print("Both answers should mention CPython, bytecode, and the Python VM.")
print("The compressed answer may omit background (creation date, popularity) -- acceptable.")

In [ ]:
# Part B -- run the full create_workflow() from src/workflow.py
# This is the canonical way to use example 143.

from src.workflow import create_workflow

workflow = create_workflow()
config = {"configurable": {"client": client, "model": "gpt-4o-mini"}}

result = workflow.invoke(
    {
        "chunks": SAMPLE_CHUNKS,
        "query": QUERY,
        "ratio": 0.4,
        "original_tokens": 0,
        "compressed_context": "",
        "compressed_tokens": 0,
        "answer": "",
    },
    config=config,
)

orig  = result["original_tokens"]
comp  = result["compressed_tokens"]
saved = orig - comp
pct   = round((1 - comp / orig) * 100) if orig else 0

print(f"Query: {QUERY}")
print()
print(f"{'Metric':<25} {'Value':>10}")
print("-" * 38)
print(f"{'Original tokens':<25} {orig:>10}")
print(f"{'Compressed tokens':<25} {comp:>10}")
print(f"{'Reduction':<25} {pct:>9}%")
print(f"{'Tokens saved':<25} {saved:>10}")
print()
print("=== Final answer (from compressed context) ===")
print(result["answer"])

---
## What's Next

- **LLMLingua paper** (https://arxiv.org/abs/2310.05736) -- token-level compression using a small LM as the scoring model; achieves 3-20x compression with minimal quality loss
- **RAG compression survey** -- how compressors integrate with retrieval pipelines (pre-retrieval, post-retrieval, and re-ranking stages)
- **Example 141 (prompt-caching)** -- once you have compressed the context, cache the compressed version so repeated queries skip the scoring step entirely
- **Example 142 (semantic-caching)** -- cache by query similarity; pair with compression to store the compressed+embedded form
- **tiktoken for exact token counting** (https://github.com/openai/tiktoken) -- essential when enforcing hard context limits or billing per token